# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` as per Croissant schema best practices.

In [ ]:
# Inspect available record sets and their schema
print("Available record sets (by @id):")
record_sets = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]
for rs_id in record_sets:
    print(f"  - {rs_id}")
    rs = dataset.get_record_set(rs_id)
    # List fields in each record set
    field_ids = [f['@id'] for f in getattr(rs, 'field', [])]
    print(f"    Fields: {field_ids}")
    # For each field, list columns
    for f in getattr(rs, 'field', []):
        print(f"      Field {f['@id']} (label: {f.get('rdfs:label','')})")
        column_ids = [c['@id'] for c in getattr(f, 'column', [])] if 'column' in f else []
        if column_ids:
            print(f"        Columns: {column_ids}")
    print()

# If record sets are not listed, try to find FileObjects and infer record sets
if not record_sets:
    print("No explicit 'recordSet' listed in metadata. Searching for tabular FileObjects...")
    # According to Croissant, tabular files can exist as distributions
    distributions = getattr(metadata, 'distribution', [])
    record_sets = []
    for dist in distributions:
        # Try to load as record set
        try:
            file_id = dist['@id']
            print(f"  - Found file distribution: {file_id}")
            record_sets.append(file_id)
        except Exception as e:
            pass
    if not record_sets:
        print("No tabular file-like record sets found in distribution.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. All references are by `@id` to follow best schema and interoperability practices.

In [ ]:
# For this dataset, we will extract data from the main tabular record set found in the distributions.
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'  # FileObject corresponding to main table

# For Croissant, loading records from the main data file (record set) using the @id
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded columns in {main_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All field references use their `@id`.

In [ ]:
# Select a numeric field for analysis
# We'll use the column with @id 'cr:coverage' (example: protein sequence coverage in percent)
numeric_field_id = 'cr:coverage'  # Put in the exact @id for coverage (%) as per schema

# Set a threshold, e.g., extract proteins with >10% coverage
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by a category, e.g., field for UniProt accession @id 'cr:accession'
    group_field_id = 'cr:accession'
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (mean values):")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in columns. Please check available fields.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We use `matplotlib` and `seaborn` for data visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field (e.g., coverage %)
plt.figure(figsize=(8,5))
if numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (Coverage %)")
    plt.xlabel("Coverage (%)")
    plt.ylabel("Count")
    plt.show()
# Boxplot by another field (e.g., peptide count, modification status if available)
if 'cr:uniquepeptides' in df.columns:  # example peptide count field
    plt.figure(figsize=(14,5))
    sns.boxplot(x=df['cr:uniquepeptides'], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by unique peptide count")
    plt.xlabel("Unique Peptides")
    plt.ylabel("Coverage (%)")
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² mass spectrometry dataset using the `mlcroissant` library, loaded data using Croissant schema `@id` references, performed filtering and normalization of protein coverage values, and visualized the main distributions. All schema entities and fields were referenced by their `@id` for full interoperability and reproducibility.

Next steps might include in-depth analysis of protein modifications, statistical modeling, or integrating with other -omics datasets.